In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os
import sys
from IPython.display import display

In [2]:
sys.path.append('../src')
from baseline_models import get_all_baselines

# Diretório para salvar os campeões
os.makedirs('../models', exist_ok=True)

# Lista dos nossos quatro cenários
motores = ['FD001', 'FD002', 'FD003', 'FD004']

# Vamos guardar o placar geral no final
resumo_campeoes = []

In [3]:
for motor in motores:
    print(f"\n{'='*50}")
    print(f"INICIANDO ARENA PARA O MOTOR: {motor}")
    print(f"{'='*50}")
    
    #Carregamento Dinâmico
    print("Carregando e achatando tensores...")
    X_train = np.load(f"../data/processed_data/X_train_{motor}.npz")['dados']
    y_train = np.load(f"../data/processed_data/y_train_{motor}.npz")['dados']
    
    X_test = np.load(f"../data/processed_data/X_test_{motor}.npz")['dados']
    y_test = np.load(f"../data/processed_data/y_test_{motor}.npz")['dados']
    
    #Achatamento (Flatten) 3D -> 2D
    n_samples_train, seq_length, n_features = X_train.shape
    X_train_2d = X_train.reshape((n_samples_train, seq_length * n_features))
    
    n_samples_test = X_test.shape[0]
    X_test_2d = X_test.reshape((n_samples_test, seq_length * n_features))
    
    print(f"Formato 2D - Treino: {X_train_2d.shape} | Teste: {X_test_2d.shape}\n")
    
    #Testando todos os modelos de baseline
    modelos = get_all_baselines()
    
    resultados_motor = []
    melhor_rmse = float('inf')
    melhor_nome_modelo = ""
    melhor_modelo = None
    
    for nome, modelo in modelos.items():
        print(f"Treinando {nome}... ", end="")
        
        #Treinamento
        modelo.fit(X_train_2d, y_train)
        
        #Previsão
        y_pred = modelo.predict(X_test_2d)
        
        #Calculando os erros
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        
        print(f"Concluído! (RMSE: {rmse:.2f})")
        
        resultados_motor.append({'Modelo': nome, 'RMSE': rmse, 'MAE': mae})
        
        #Atualizando o campeão
        if rmse < melhor_rmse:
            melhor_rmse = rmse
            melhor_nome_modelo = nome
            melhor_modelo = modelo
            
    #Placar do Motor Atual
    df_resultados = pd.DataFrame(resultados_motor).sort_values(by='RMSE').reset_index(drop=True)
    print(f"\nPlacar Final - {motor}:")
    display(df_resultados)
    
    nome_arquivo = melhor_nome_modelo.lower().replace(" ", "_")
    caminho_salvamento = f'../models/best_baseline_{motor.lower()}_{nome_arquivo}.joblib'
    joblib.dump(melhor_modelo, caminho_salvamento)
    print(f"Modelo campeão salvo em: {caminho_salvamento}")
    
    # Adicionando à lista geral para exibição final
    resumo_campeoes.append({'Motor': motor, 'Campeão': melhor_nome_modelo, 'RMSE': melhor_rmse})

# Exibição do resumo final de todos os motores
print("\n" + "="*50)
print("RESUMO GERAL DOS CAMPEÕES")
print("="*50)
display(pd.DataFrame(resumo_campeoes))


INICIANDO ARENA PARA O MOTOR: FD001
Carregando e achatando tensores...
Formato 2D - Treino: (17731, 570) | Teste: (10196, 570)

Treinando Ridge... Concluído! (RMSE: 16.80)
Treinando KNN... Concluído! (RMSE: 21.11)
Treinando SVR... Concluído! (RMSE: 15.23)
Treinando Random Forest... Concluído! (RMSE: 17.99)
Treinando XGBoost... Concluído! (RMSE: 16.21)

Placar Final - FD001:


,Modelo,RMSE,MAE
0,SVR,15.225115,11.817452
1,XGBoost,16.208354,12.481792
2,Ridge,16.800015,13.677403
3,Random Forest,17.993880,14.024409
4,KNN,21.107949,16.209082


Modelo campeão salvo em: ../models/best_baseline_fd001_svr.joblib

INICIANDO ARENA PARA O MOTOR: FD002
Carregando e achatando tensores...
Formato 2D - Treino: (46219, 720) | Teste: (26511, 720)

Treinando Ridge... Concluído! (RMSE: 18.71)
Treinando KNN... Concluído! (RMSE: 24.36)
Treinando SVR... Concluído! (RMSE: 17.90)
Treinando Random Forest... Concluído! (RMSE: 19.82)
Treinando XGBoost... Concluído! (RMSE: 18.17)

Placar Final - FD002:


,Modelo,RMSE,MAE
0,SVR,17.899701,14.042678
1,XGBoost,18.170295,14.229383
2,Ridge,18.708536,15.300395
3,Random Forest,19.823802,16.048472
4,KNN,24.355496,19.407929


Modelo campeão salvo em: ../models/best_baseline_fd002_svr.joblib

INICIANDO ARENA PARA O MOTOR: FD003
Carregando e achatando tensores...
Formato 2D - Treino: (21820, 600) | Teste: (13696, 600)

Treinando Ridge... Concluído! (RMSE: 15.89)
Treinando KNN... Concluído! (RMSE: 18.55)
Treinando SVR... Concluído! (RMSE: 13.10)
Treinando Random Forest... Concluído! (RMSE: 14.52)
Treinando XGBoost... Concluído! (RMSE: 12.84)

Placar Final - FD003:


,Modelo,RMSE,MAE
0,XGBoost,12.835161,8.598989
1,SVR,13.100614,9.243997
2,Random Forest,14.523682,10.201283
3,Ridge,15.886918,12.278706
4,KNN,18.549252,12.373189


Modelo campeão salvo em: ../models/best_baseline_fd003_xgboost.joblib

INICIANDO ARENA PARA O MOTOR: FD004
Carregando e achatando tensores...
Formato 2D - Treino: (54028, 720) | Teste: (34092, 720)

Treinando Ridge... Concluído! (RMSE: 18.27)
Treinando KNN... Concluído! (RMSE: 21.27)
Treinando SVR... Concluído! (RMSE: 15.85)
Treinando Random Forest... Concluído! (RMSE: 16.78)
Treinando XGBoost... Concluído! (RMSE: 15.67)

Placar Final - FD004:


,Modelo,RMSE,MAE
0,XGBoost,15.671385,10.474434
1,SVR,15.853382,11.182887
2,Random Forest,16.782703,11.767032
3,Ridge,18.274336,14.246556
4,KNN,21.273599,14.722293


Modelo campeão salvo em: ../models/best_baseline_fd004_xgboost.joblib

RESUMO GERAL DOS CAMPEÕES


,Motor,Campeão,RMSE
0,FD001,SVR,15.225115
1,FD002,SVR,17.899701
2,FD003,XGBoost,12.835161
3,FD004,XGBoost,15.671385
